# Debug Existing RAG Pipeline Modules

Use this notebook to test the modules implemented so far:

1. `DocumentLoader`
2. `VietnameseTextPreprocessor`
3. `TextSplitter`
4. `EmbeddingModel`
5. `VectorStore`

Run the cells from top to bottom. The embedding and ChromaDB cells require the full dependencies from `requirements.txt`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## 1. Create A Small Vietnamese TXT Sample

In [ ]:
sample_path = PROJECT_ROOT / "data" / "samples" / "debug_vietnamese.txt"
sample_path.parent.mkdir(parents=True, exist_ok=True)

sample_path.write_text(
    "Sinh viên phải hoàn\nthành tối thiểu 65 % tổng số tín chỉ để được đăng ký thực tập.\n\n"
    "Sinh viên cần có điểm trung bình tích lũy đạt yêu cầu của khoa.\n\n"
    "Những câu hỏi không có trong tài liệu cần được hệ thống từ chối trả lời.",
    encoding="utf-8",
)

sample_path

## 2. Test DocumentLoader

In [ ]:
from app.document_loader import DocumentLoader

loader = DocumentLoader()
documents = loader.load(str(sample_path), sample_path.name, "debug-doc")
documents

## 3. Test VietnameseTextPreprocessor

In [ ]:
from app.text_preprocessor import VietnameseTextPreprocessor

preprocessor = VietnameseTextPreprocessor()
processed_documents = []

for document in documents:
    processed_documents.append(
        {
            "text": preprocessor.clean(document["text"]),
            "metadata": document["metadata"],
        }
    )

processed_documents

## 4. Test TextSplitter

In [ ]:
from app.text_splitter import TextSplitter

splitter = TextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split(processed_documents)

for chunk in chunks:
    print(chunk["chunk_id"])
    print(chunk["metadata"])
    print(chunk["text"])
    print("-" * 80)

## 5. Optional: Test EmbeddingModel

This downloads/loads the multilingual sentence-transformers model if it is not already cached.

In [ ]:
from app.config import get_config
from app.embedding_model import EmbeddingModel

config = get_config()
embedder = EmbeddingModel(config.embedding_model)
embeddings = embedder.encode([chunk["text"] for chunk in chunks])

len(embeddings), len(embeddings[0]) if embeddings else 0

## 6. Optional: Test VectorStore

In [ ]:
from app.vector_store import VectorStore

debug_store = VectorStore(PROJECT_ROOT / "vector_db" / "debug", "debug_document_qa")
debug_store.reset()
debug_store.add_chunks(chunks, embeddings)

query = preprocessor.clean_query("Sinh viên cần bao nhiêu tín chỉ để đăng ký thực tập?")
query_embedding = embedder.encode([query])[0]
results = debug_store.search(query_embedding, top_k=3)
results